# Guardrails for the Event-Planning Multi-Agent System

This notebook adds a **guardrail layer** around the four single-purpose agents from
`Task.ipynb` (`VenueAgent`, `CateringLogisticsAgent`, `AVVendorCoordinationAgent`,
`AgendaAssemblyAgent`) to protect the system against misuse, low-quality inputs, hallucinated
outputs, and runaway cost.

**Deliverables in this notebook**

1. **Input guardrail** - an LLM-as-classifier (`InputGuardrail`) that screens the raw user
   request *before any agent is called* and blocks it if it is off-topic or harmful.
2. **Output guardrail** - deterministic, rule-based checks (`OutputGuardrail`) that catch
   hallucinated details in an assembled plan, covering at least two factual-consistency rules:
   - *Timeline feasibility*: no overlapping schedule entries, and no setup/teardown activity
     scheduled after the event itself starts or with an implausible lead time.
   - *Venue/location consistency*: the selected venue's name/area must not reference a city
     inconsistent with the requester's actual `location`.
3. **Cost/rate-limit guard** - a `CostCapGuard` that wraps the LLM client, estimates token spend
   *before* every call, and aborts the pipeline gracefully with a user-facing message the moment
   projected spend would exceed a configured cap.
4. A `GuardedEventPipeline` that wires all three guards around the four agents, demonstrated
   end-to-end across four scenarios: a clean pass, a blocked (off-topic) request, a cost-cap
   abort, and an output flagged for human review due to hallucinated details.

As in the earlier notebooks, every agent and guard depends only on the duck-typed `LLMClient`
protocol, so the whole notebook runs deterministically with no network access or API credentials
via scripted fake backends. Swap in `AzureOpenAIClient` for production use.


In [1]:
import json
import os
import sys
from collections import deque
from enum import Enum
from typing import Any, Dict, List, Literal, Optional, Protocol, Tuple, Type

import pandas as pd
from pydantic import BaseModel, Field

# Make the repo-root `init_creds.py` importable - only needed if/when `AzureOpenAIClient` below
# is actually instantiated.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


class EventType(str, Enum):
    """Shared event taxonomy used as input by every agent (same as `Task.ipynb`)."""

    CONFERENCE = "conference"
    WEDDING = "wedding"
    CORPORATE_OFFSITE = "corporate_offsite"
    PRODUCT_LAUNCH = "product_launch"
    GALA = "gala"
    WORKSHOP = "workshop"
    OTHER = "other"


class LLMClient(Protocol):
    """Minimal interface every agent - and every guard - depends on (same contract as
    `Task.ipynb`)."""

    def generate(
        self,
        system_prompt: str,
        user_payload: Dict[str, Any],
        schema_name: str,
        json_schema: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a dict that should validate against `json_schema`."""
        ...


class AzureOpenAIClient:
    """Production LLM backend: Azure OpenAI chat completions with structured JSON output.

    Credentials are resolved lazily (via the repo's `init_creds.py` helpers) only when an
    instance is actually constructed, so importing/using this module never requires network
    access or secrets - that is reserved for the scripted fakes below.
    """

    def __init__(self, model: str = "gpt-4o-mini", api_version: str = "2025-04-01-preview"):
        from openai import AzureOpenAI
        import init_creds as creds

        self._client = AzureOpenAI(
            api_key=creds.get_api_key(),
            azure_endpoint=creds.get_endpoint(),
            api_version=api_version,
        )
        self._model = model

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_payload)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": schema_name, "schema": json_schema, "strict": False},
            },
        )
        return json.loads(response.choices[0].message.content)


class ScriptedLLMClient:
    """Deterministic fake backend: returns one canned response per call, popped off a scripted
    queue in order (same philosophy as `Task2.ipynb`'s `ScriptedLLMClient`). Used for the
    guardrail classifiers below, where each call represents a different, pre-determined verdict."""

    def __init__(self, responses: List[Dict[str, Any]]):
        self._responses = deque(responses)
        self.calls: List[Dict[str, Any]] = []

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        self.calls.append({"schema_name": schema_name, "user_payload": user_payload})
        if not self._responses:
            raise RuntimeError("ScriptedLLMClient ran out of canned responses")
        return self._responses.popleft()


class KeyedFakeLLMClient:
    """Deterministic fake backend keyed by `schema_name` (= each agent's `name`), so a single
    instance can back all four different agents in the guarded-pipeline demos below (same
    philosophy as `Task3.ipynb`'s `FakeLLMClient`)."""

    def __init__(self, canned_responses: Dict[str, Dict[str, Any]]):
        self.canned_responses = canned_responses
        self.calls: List[Dict[str, Any]] = []

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        self.calls.append({"schema_name": schema_name, "user_payload": user_payload})
        if schema_name not in self.canned_responses:
            raise KeyError(f"No canned response registered for '{schema_name}'")
        return self.canned_responses[schema_name]


## 1. The four specialist agents (reused from `Task.ipynb`)

Same `BaseAgent` scaffolding and the same four agent classes/schemas as the earlier notebooks,
plus a trimmed `EventRequest` (the single user-facing request the guarded pipeline accepts) -
repeated here so this notebook is self-contained and runnable on its own.


In [2]:
class BaseAgent:
    """Shared scaffolding for single-purpose, structured-output agents."""

    name: str
    system_prompt: str
    input_model: Type[BaseModel]
    output_model: Type[BaseModel]

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def run(self, request):
        if not isinstance(request, self.input_model):
            request = self.input_model.model_validate(request)
        payload = request.model_dump(mode="json")
        raw_output = self._llm.generate(
            system_prompt=self.system_prompt,
            user_payload=payload,
            schema_name=self.name,
            json_schema=self.output_model.model_json_schema(),
        )
        return self.output_model.model_validate(raw_output)


class VenueRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    location: str = Field(min_length=1)
    event_date: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class VenueOption(BaseModel):
    name: str
    area: str
    capacity: int = Field(gt=0)
    estimated_cost_usd: float = Field(ge=0)
    amenities: List[str] = Field(default_factory=list)
    rationale: str


class VenueRecommendation(BaseModel):
    recommended_venues: List[VenueOption] = Field(min_length=1)
    assumptions: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class VenueAgent(BaseAgent):
    name = "venue_recommendation_agent"
    input_model = VenueRequest
    output_model = VenueRecommendation
    system_prompt = (
        "You are a single-purpose venue recommendation specialist for live events. Propose "
        "2-4 realistic candidate venues from headcount, budget, location, and event type, all "
        "genuinely located in the requested location. Never recommend a venue with capacity "
        "below the requested headcount. Respond ONLY with the structured JSON object defined by "
        "the provided schema."
    )


class CateringRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(default_factory=list)
    service_style_preference: Optional[
        Literal["plated", "buffet", "family_style", "food_stations", "cocktail"]
    ] = None


class MenuItem(BaseModel):
    course: str
    description: str
    dietary_tags: List[str] = Field(default_factory=list)


class CateringPlan(BaseModel):
    service_style: str
    menu_items: List[MenuItem] = Field(min_length=1)
    staffing_notes: str
    estimated_cost_usd: float = Field(ge=0)
    cost_per_head_usd: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class CateringLogisticsAgent(BaseAgent):
    name = "catering_logistics_agent"
    input_model = CateringRequest
    output_model = CateringPlan
    system_prompt = (
        "You are a single-purpose catering and food-service logistics specialist. Propose ONE "
        "coherent catering plan covering every dietary restriction supplied. Respond ONLY with "
        "the structured JSON object defined by the provided schema."
    )


class AVRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    venue_type: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class AVSetupItem(BaseModel):
    category: str
    description: str
    quantity: int = Field(ge=1)


class VendorSuggestion(BaseModel):
    vendor_role: str
    notes: str


class AVPlan(BaseModel):
    av_setup: List[AVSetupItem] = Field(min_length=1)
    recommended_vendors: List[VendorSuggestion] = Field(min_length=1)
    estimated_cost_usd: float = Field(ge=0)
    setup_lead_time_hours: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class AVVendorCoordinationAgent(BaseAgent):
    name = "av_vendor_coordination_agent"
    input_model = AVRequest
    output_model = AVPlan
    system_prompt = (
        "You are a single-purpose audiovisual and vendor-coordination specialist. Propose ONE "
        "AV setup plan with a realistic setup_lead_time_hours. Respond ONLY with the structured "
        "JSON object defined by the provided schema."
    )


class AgendaRequest(BaseModel):
    event_name: str
    event_date: str
    start_time: str
    end_time: str
    venue: VenueRecommendation
    catering: CateringPlan
    av: AVPlan


class RunOfShowItem(BaseModel):
    start_time: str
    end_time: str
    activity: str
    owner: str
    location: Optional[str] = None
    notes: Optional[str] = None


class RunOfShow(BaseModel):
    event_name: str
    event_date: str
    selected_venue: str
    run_of_show: List[RunOfShowItem] = Field(min_length=1)
    overall_notes: List[str] = Field(default_factory=list)


class AgendaAssemblyAgent(BaseAgent):
    name = "agenda_assembly_agent"
    input_model = AgendaRequest
    output_model = RunOfShow
    system_prompt = (
        "You are a single-purpose agenda-assembly specialist. Choose exactly ONE venue from "
        "`venue.recommended_venues` and sequence catering + AV + main activities into a single, "
        "non-overlapping chronological run-of-show, with AV setup finishing no later than the "
        "event's start_time. Respond ONLY with the structured JSON object defined by the "
        "provided schema."
    )


VENUE_BUDGET_SHARE = 0.45
CATERING_BUDGET_SHARE = 0.35
AV_BUDGET_SHARE = 0.20


class EventRequest(BaseModel):
    """The single user-facing request the guarded pipeline accepts."""

    event_name: str
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    location: str
    event_date: str = Field(description="ISO date (YYYY-MM-DD)")
    start_time: str = Field(description="24h HH:MM")
    end_time: str = Field(description="24h HH:MM")
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(default_factory=list)
    special_requirements: List[str] = Field(default_factory=list)


## 2. Input guardrail - block off-topic or harmful requests before any agent runs

`InputGuardrail` is an LLM-as-classifier that screens the **raw** user request into exactly one
category - `on_topic`, `off_topic`, or `harmful` - before `EventRequest` is even parsed or any
specialist agent is invoked. Only `on_topic` requests are allowed through; everything else is
rejected with the judge's own stated reason.


In [3]:
class GuardrailVerdict(BaseModel):
    category: Literal["on_topic", "off_topic", "harmful"]
    reason: str = Field(description="Brief justification for the classification")


ALLOWED_INPUT_CATEGORY = "on_topic"


class InputGuardrail:
    """LLM-as-classifier: screens the raw user request BEFORE any specialist agent is invoked,
    blocking anything off-topic (unrelated to event planning) or harmful (illegal/fraudulent/
    abusive intent disguised as an event-planning request)."""

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def _system_prompt(self) -> str:
        return (
            "You are a content-safety classifier gating access to an event-planning multi-agent "
            "system. Classify the user's raw request into exactly one category:\n"
            "- 'on_topic': a legitimate request to plan, source venues/catering/AV for, or "
            "schedule a live event (conference, wedding, offsite, product launch, gala, "
            "workshop, etc.).\n"
            "- 'off_topic': unrelated to event planning (e.g. general trivia, recipes, coding "
            "help, unrelated small talk) - not harmful, just out of scope for this system.\n"
            "- 'harmful': seeks to use the event-planning system to facilitate illegal activity, "
            "fraud, harassment, or physical harm, even if disguised as an event request (e.g. "
            "using a fake 'event' to collect attendees' payment details without consent).\n"
            "Respond ONLY with the structured JSON object defined by the provided schema."
        )

    def check(self, raw_request: str) -> GuardrailVerdict:
        raw = self._llm.generate(
            system_prompt=self._system_prompt(),
            user_payload={"raw_request": raw_request},
            schema_name="input_guardrail",
            json_schema=GuardrailVerdict.model_json_schema(),
        )
        return GuardrailVerdict.model_validate(raw)

    def is_allowed(self, verdict: GuardrailVerdict) -> bool:
        return verdict.category == ALLOWED_INPUT_CATEGORY


In [4]:
input_guardrail_demo_llm = ScriptedLLMClient(responses=[
    {
        "category": "on_topic",
        "reason": "Requests sourcing a venue, catering, and AV for a 300-person product launch - core event-planning scope.",
    },
    {
        "category": "off_topic",
        "reason": "Asks for a banana bread recipe, which has nothing to do with planning a live event.",
    },
    {
        "category": "harmful",
        "reason": "Describes using a fake 'conference' as a front to collect attendees' credit card numbers without consent - fraud.",
    },
])
input_guardrail_demo = InputGuardrail(input_guardrail_demo_llm)

demo_requests = [
    "We need help organizing a 300-person product launch in Denver next spring, including venue, catering, and AV.",
    "Can you give me a good recipe for banana bread?",
    "I want to plan a 'conference' that's actually just a front to collect attendees' credit card numbers without them knowing.",
]

for text in demo_requests:
    verdict = input_guardrail_demo.check(text)
    decision = "ALLOWED" if input_guardrail_demo.is_allowed(verdict) else "BLOCKED"
    print(f"[{decision:>7}] category={verdict.category:<10} reason={verdict.reason}")

assert input_guardrail_demo.is_allowed(GuardrailVerdict(category="on_topic", reason=""))
assert not input_guardrail_demo.is_allowed(GuardrailVerdict(category="off_topic", reason=""))
assert not input_guardrail_demo.is_allowed(GuardrailVerdict(category="harmful", reason=""))
print("\nInput guardrail allow/block routing checks passed.")


[ALLOWED] category=on_topic   reason=Requests sourcing a venue, catering, and AV for a 300-person product launch - core event-planning scope.
[BLOCKED] category=off_topic  reason=Asks for a banana bread recipe, which has nothing to do with planning a live event.
[BLOCKED] category=harmful    reason=Describes using a fake 'conference' as a front to collect attendees' credit card numbers without consent - fraud.

Input guardrail allow/block routing checks passed.


## 3. Output guardrail - catch hallucinated details with deterministic rules

`OutputGuardrail` checks an **assembled** plan (the venue recommendation, the AV plan, and the
final run-of-show) against two factual-consistency rule groups. These are plain Python checks,
not another LLM call - cheaper, deterministic, and immune to the judge itself hallucinating a
pass:

1. **Timeline feasibility** - every run-of-show entry must end after it starts; no two entries at
   the *same location* may overlap; and any setup/AV entry must finish **before** the event's own
   `start_time` with a plausible lead time (more than `MAX_SETUP_LEAD_HOURS` implies a hallucinated
   "16-hour setup" rather than a real one).
2. **Venue/location consistency** - the `selected_venue` must be one of the venues actually
   proposed by `VenueAgent`, and its name/area must not reference a city inconsistent with the
   requester's real `location` (e.g. a "Miami" venue for an "Austin, TX" request) - a classic
   sign of a fabricated venue.


In [5]:
# Small gazetteer used only to detect a venue name/area referencing a DIFFERENT known city than
# the one requested - not an exhaustive geocoder, just enough to catch an obviously fabricated
# cross-city venue in this demo.
GAZETTEER: Dict[str, str] = {
    "austin": "TX", "chicago": "IL", "miami": "FL", "denver": "CO", "new york": "NY",
    "los angeles": "CA", "san francisco": "CA", "seattle": "WA", "boston": "MA", "houston": "TX",
}


def _primary_city(location: str) -> Optional[str]:
    lowered = location.lower()
    for city in GAZETTEER:
        if city in lowered:
            return city
    return None


def _to_minutes(hhmm: str) -> int:
    hours, minutes = hhmm.split(":")
    return int(hours) * 60 + int(minutes)


class OutputGuardrailResult(BaseModel):
    passed: bool
    violations: List[str] = Field(default_factory=list)


class OutputGuardrail:
    """Deterministic, rule-based checks that catch hallucinated details in an assembled plan
    before it is shown to the user."""

    MAX_SETUP_LEAD_HOURS = 12.0  # a setup window longer than this is almost certainly hallucinated

    def check(
        self,
        request: EventRequest,
        venue: VenueRecommendation,
        av: AVPlan,
        run_of_show: RunOfShow,
    ) -> OutputGuardrailResult:
        violations: List[str] = []
        violations.extend(self._check_timeline_feasibility(request, av, run_of_show))
        violations.extend(self._check_venue_location_consistency(request, venue, run_of_show))
        return OutputGuardrailResult(passed=not violations, violations=violations)

    def _check_timeline_feasibility(
        self, request: EventRequest, av: AVPlan, run_of_show: RunOfShow
    ) -> List[str]:
        violations: List[str] = []
        event_start = _to_minutes(request.start_time)

        for item in run_of_show.run_of_show:
            start, end = _to_minutes(item.start_time), _to_minutes(item.end_time)
            if end <= start:
                violations.append(
                    f"Run-of-show entry '{item.activity}' ends ({item.end_time}) at or before it "
                    f"starts ({item.start_time})."
                )
            is_setup_entry = "setup" in item.activity.lower() or "av crew" in item.owner.lower()
            if is_setup_entry:
                if start > event_start:
                    violations.append(
                        f"Setup entry '{item.activity}' is scheduled to start ({item.start_time}) "
                        f"AFTER the event's own start time ({request.start_time}) - impossible."
                    )
                lead_hours = (event_start - start) / 60
                if lead_hours > self.MAX_SETUP_LEAD_HOURS:
                    violations.append(
                        f"Setup entry '{item.activity}' implies a {lead_hours:.1f}-hour lead time "
                        f"(vs. av.setup_lead_time_hours={av.setup_lead_time_hours}) - implausible "
                        "for a live event and likely hallucinated."
                    )

        by_location: Dict[str, List[Tuple[int, int, str]]] = {}
        for item in run_of_show.run_of_show:
            if item.location:
                by_location.setdefault(item.location, []).append(
                    (_to_minutes(item.start_time), _to_minutes(item.end_time), item.activity)
                )
        for location, entries in by_location.items():
            entries.sort()
            for (s1, e1, a1), (s2, e2, a2) in zip(entries, entries[1:]):
                if s2 < e1:
                    violations.append(
                        f"'{a1}' and '{a2}' overlap at location '{location}' - both can't occupy "
                        "the same space at the same time."
                    )
        return violations

    def _check_venue_location_consistency(
        self, request: EventRequest, venue: VenueRecommendation, run_of_show: RunOfShow
    ) -> List[str]:
        requested_city = _primary_city(request.location)
        if requested_city is None:
            return []  # unrecognized city - nothing to cross-check against

        selected = next(
            (v for v in venue.recommended_venues if v.name == run_of_show.selected_venue), None
        )
        if selected is None:
            return [
                f"selected_venue '{run_of_show.selected_venue}' does not match any venue in "
                "venue.recommended_venues - likely a fabricated venue name."
            ]

        text = f"{selected.name} {selected.area}".lower()
        violations = [
            f"Venue '{selected.name}' (area '{selected.area}') references '{city.title()}', "
            f"which is inconsistent with the requested location ('{request.location}') - likely "
            "a hallucinated/fabricated venue."
            for city in GAZETTEER
            if city != requested_city and city in text
        ]
        return violations


In [6]:
output_guardrail = OutputGuardrail()

demo_request = EventRequest(
    event_name="Innovate Summit", event_type=EventType.CONFERENCE, headcount=120,
    budget_usd=20000, location="Austin, TX", event_date="2026-09-01",
    start_time="09:00", end_time="15:00", meal_period="lunch",
)
demo_venue = VenueRecommendation(recommended_venues=[
    VenueOption(name="Riverside Conference Center", area="Downtown Austin", capacity=200,
                estimated_cost_usd=9000.0, amenities=["wifi"], rationale="Fits brief."),
])
demo_av = AVPlan(
    av_setup=[AVSetupItem(category="audio", description="Line array speakers", quantity=2)],
    recommended_vendors=[VendorSuggestion(vendor_role="AV production company", notes="Local vendor.")],
    estimated_cost_usd=3500.0, setup_lead_time_hours=2.0,
)

# Scenario 1: clean plan - should PASS both rule groups.
clean_run_of_show = RunOfShow(
    event_name="Innovate Summit", event_date="2026-09-01", selected_venue="Riverside Conference Center",
    run_of_show=[
        RunOfShowItem(start_time="07:00", end_time="09:00", activity="AV setup and sound check", owner="AV crew", location="Ballroom"),
        RunOfShowItem(start_time="09:00", end_time="12:00", activity="Keynote and breakout sessions", owner="Host", location="Ballroom"),
        RunOfShowItem(start_time="12:00", end_time="13:00", activity="Buffet lunch service", owner="Catering team", location="Ballroom"),
    ],
)

# Scenario 2: timeline hallucination - AV setup starts AFTER the event and overlaps the keynote.
timeline_violation_run_of_show = RunOfShow(
    event_name="Innovate Summit", event_date="2026-09-01", selected_venue="Riverside Conference Center",
    run_of_show=[
        RunOfShowItem(start_time="09:30", end_time="10:00", activity="AV setup and sound check", owner="AV crew", location="Ballroom"),
        RunOfShowItem(start_time="09:00", end_time="12:00", activity="Keynote and breakout sessions", owner="Host", location="Ballroom"),
    ],
)

# Scenario 3: fabricated venue - selected_venue references Miami for an Austin, TX request.
location_violation_venue = VenueRecommendation(recommended_venues=[
    VenueOption(name="South Beach Miami Pavilion", area="South Beach, Miami", capacity=150,
                estimated_cost_usd=9000.0, amenities=["ocean view"], rationale="Waterfront option."),
])
location_violation_run_of_show = RunOfShow(
    event_name="Innovate Summit", event_date="2026-09-01", selected_venue="South Beach Miami Pavilion",
    run_of_show=[
        RunOfShowItem(start_time="07:00", end_time="09:00", activity="AV setup", owner="AV crew", location="Ballroom"),
        RunOfShowItem(start_time="09:00", end_time="12:00", activity="Keynote", owner="Host", location="Ballroom"),
    ],
)

for label, venue, run_of_show in [
    ("Scenario 1 (clean plan)", demo_venue, clean_run_of_show),
    ("Scenario 2 (timeline hallucination)", demo_venue, timeline_violation_run_of_show),
    ("Scenario 3 (venue/location hallucination)", location_violation_venue, location_violation_run_of_show),
]:
    result = output_guardrail.check(demo_request, venue, demo_av, run_of_show)
    print(f"--- {label}: passed={result.passed} ---")
    for v in result.violations:
        print(f"  - {v}")

assert output_guardrail.check(demo_request, demo_venue, demo_av, clean_run_of_show).passed
assert not output_guardrail.check(demo_request, demo_venue, demo_av, timeline_violation_run_of_show).passed
assert not output_guardrail.check(demo_request, location_violation_venue, demo_av, location_violation_run_of_show).passed
print("\nOutput guardrail pass/violation checks passed.")


--- Scenario 1 (clean plan): passed=True ---
--- Scenario 2 (timeline hallucination): passed=False ---
  - Setup entry 'AV setup and sound check' is scheduled to start (09:30) AFTER the event's own start time (09:00) - impossible.
  - 'Keynote and breakout sessions' and 'AV setup and sound check' overlap at location 'Ballroom' - both can't occupy the same space at the same time.
--- Scenario 3 (venue/location hallucination): passed=False ---
  - Venue 'South Beach Miami Pavilion' (area 'South Beach, Miami') references 'Miami', which is inconsistent with the requested location ('Austin, TX') - likely a hallucinated/fabricated venue.

Output guardrail pass/violation checks passed.


## 4. Cost/rate-limit guard - abort before overspending

`CostCapGuard` wraps any `LLMClient` and estimates the token cost of a call **before** it is
made (input tokens from the actual prompt/payload, output tokens from a conservative fixed
estimate). If the *projected cumulative* spend for the pipeline run would exceed `max_cost_usd`,
it raises `CostCapExceededError` and the inner client is never called - no money or quota is
actually spent on the call that would have breached the cap. After a call succeeds, it updates
its running totals using the *actual* response size, so later calls in the same run are judged
against real spend so far.

The token estimate (`len(text) // 4`) is a dependency-free approximation; swap in `tiktoken` for
exact counts in production.


In [7]:
# Illustrative gpt-4o-mini-class pricing (USD per 1K tokens) - swap for your provider's real rates.
PRICE_PER_1K_INPUT_TOKENS_USD = 0.00015
PRICE_PER_1K_OUTPUT_TOKENS_USD = 0.0006
ASSUMED_OUTPUT_TOKENS = 600  # conservative pre-call estimate of a structured-JSON agent response


def estimate_tokens(text: str) -> int:
    """Rough, dependency-free token estimate (~4 characters per token) - good enough for
    order-of-magnitude cost budgeting; swap in `tiktoken` for exact counts in production."""
    return max(1, len(text) // 4)


class CostCapExceededError(Exception):
    """Raised BEFORE a call is made that would push cumulative estimated spend over the cap."""


class CostCapGuard:
    """Wraps any `LLMClient` and aborts before making a call that would push this pipeline run's
    cumulative estimated spend over `max_cost_usd` - a rate-limit/cost-cap guard."""

    def __init__(self, inner: LLMClient, max_cost_usd: float, assumed_output_tokens: int = ASSUMED_OUTPUT_TOKENS):
        self._inner = inner
        self.max_cost_usd = max_cost_usd
        self.assumed_output_tokens = assumed_output_tokens
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost_usd = 0.0
        self.call_count = 0

    @staticmethod
    def _cost(input_tokens: int, output_tokens: int) -> float:
        return (
            (input_tokens / 1000) * PRICE_PER_1K_INPUT_TOKENS_USD
            + (output_tokens / 1000) * PRICE_PER_1K_OUTPUT_TOKENS_USD
        )

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        input_tokens = estimate_tokens(system_prompt) + estimate_tokens(json.dumps(user_payload))
        projected_cost = self.total_cost_usd + self._cost(input_tokens, self.assumed_output_tokens)
        if projected_cost > self.max_cost_usd:
            raise CostCapExceededError(
                f"We're sorry - completing this request would cost an estimated ${projected_cost:.5f}, "
                f"which exceeds the ${self.max_cost_usd:.5f} budget cap for this request. The request "
                f"has been stopped before calling '{schema_name}' to avoid overspending. Please "
                "simplify the request (e.g. fewer special requirements) or contact support to raise "
                "the budget cap."
            )

        raw = self._inner.generate(system_prompt, user_payload, schema_name, json_schema)

        output_tokens = estimate_tokens(json.dumps(raw))
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cost_usd += self._cost(input_tokens, output_tokens)
        self.call_count += 1
        return raw


In [8]:
demo_backend = KeyedFakeLLMClient({
    "venue_recommendation_agent": {
        "recommended_venues": [
            {"name": "Riverside Conference Center", "area": "Downtown Austin", "capacity": 200,
             "estimated_cost_usd": 9000.0, "amenities": ["wifi"], "rationale": "Fits brief."}
        ],
        "assumptions": [],
    }
})

# Generous cap - the call goes through normally.
generous_guard = CostCapGuard(demo_backend, max_cost_usd=1.00)
generous_guard.generate("system prompt", {"headcount": 120}, "venue_recommendation_agent", {})
print(f"Generous cap: call_count={generous_guard.call_count}, total_cost_usd={generous_guard.total_cost_usd:.6f}")

# Tiny cap - the SAME call is rejected before the inner client is ever invoked.
tiny_guard = CostCapGuard(demo_backend, max_cost_usd=0.000001)
try:
    tiny_guard.generate("system prompt", {"headcount": 120}, "venue_recommendation_agent", {})
    raise AssertionError("expected CostCapExceededError to be raised")
except CostCapExceededError as exc:
    print(f"\nTiny cap aborted gracefully:\n{exc}")

assert tiny_guard.call_count == 0, "no call should have reached the inner client once the cap was exceeded"
print("\nCost-cap guard checks passed.")


Generous cap: call_count=1, total_cost_usd=0.000032

Tiny cap aborted gracefully:
We're sorry - completing this request would cost an estimated $0.00036, which exceeds the $0.00000 budget cap for this request. The request has been stopped before calling 'venue_recommendation_agent' to avoid overspending. Please simplify the request (e.g. fewer special requirements) or contact support to raise the budget cap.

Cost-cap guard checks passed.


## 5. The guarded pipeline - wiring all three guards around the four agents

`GuardedEventPipeline.run` takes the user's raw free-text request alongside the parsed
`EventRequest`, and applies the guards in order:

1. **Input guardrail** - if the raw request is not `on_topic`, stop immediately; no agent is ever
   called.
2. **Agents (cost-capped)** - venue -> catering -> AV -> agenda, each call going through a shared
   `CostCapGuard` so spend accumulates across the whole request; if any call would breach the
   cap, the pipeline aborts right there with the guard's user-facing message.
3. **Output guardrail** - once a plan is assembled, run the factual-consistency checks; if any
   rule fails, the plan is flagged for human review rather than returned as final.

A single `PipelineResult.status` tells the caller exactly how the request was handled:
`completed`, `blocked_input`, `aborted_cost_cap`, or `flagged_output`.


In [9]:
class PipelineResult(BaseModel):
    status: Literal["completed", "blocked_input", "aborted_cost_cap", "flagged_output"]
    message: str
    plan: Optional[RunOfShow] = None
    violations: List[str] = Field(default_factory=list)
    trace: List[str] = Field(default_factory=list)


class GuardedEventPipeline:
    """Runs the four specialist agents behind the input guardrail, a shared cost-cap guard, and
    the output guardrail."""

    def __init__(
        self,
        venue_agent: VenueAgent,
        catering_agent: CateringLogisticsAgent,
        av_agent: AVVendorCoordinationAgent,
        agenda_agent: AgendaAssemblyAgent,
        input_guardrail: InputGuardrail,
        output_guardrail: OutputGuardrail,
    ):
        self._venue_agent = venue_agent
        self._catering_agent = catering_agent
        self._av_agent = av_agent
        self._agenda_agent = agenda_agent
        self._input_guardrail = input_guardrail
        self._output_guardrail = output_guardrail

    def run(self, raw_user_text: str, request: EventRequest) -> PipelineResult:
        trace: List[str] = []

        verdict = self._input_guardrail.check(raw_user_text)
        trace.append(f"input_guardrail: category={verdict.category} reason={verdict.reason!r}")
        if not self._input_guardrail.is_allowed(verdict):
            return PipelineResult(
                status="blocked_input",
                message=f"Request rejected before any agent ran: {verdict.reason}",
                trace=trace,
            )

        try:
            venue = self._venue_agent.run(VenueRequest(
                event_type=request.event_type, headcount=request.headcount,
                budget_usd=request.budget_usd * VENUE_BUDGET_SHARE, location=request.location,
                event_date=request.event_date, special_requirements=request.special_requirements,
            ))
            trace.append("venue_agent: completed")

            catering = self._catering_agent.run(CateringRequest(
                event_type=request.event_type, headcount=request.headcount,
                budget_usd=request.budget_usd * CATERING_BUDGET_SHARE, meal_period=request.meal_period,
                dietary_restrictions=request.dietary_restrictions,
            ))
            trace.append("catering_agent: completed")

            selected = venue.recommended_venues[0]
            av = self._av_agent.run(AVRequest(
                event_type=request.event_type, headcount=request.headcount,
                budget_usd=request.budget_usd * AV_BUDGET_SHARE, venue_type=selected.area,
                special_requirements=request.special_requirements,
            ))
            trace.append("av_agent: completed")

            run_of_show = self._agenda_agent.run(AgendaRequest(
                event_name=request.event_name, event_date=request.event_date,
                start_time=request.start_time, end_time=request.end_time,
                venue=venue, catering=catering, av=av,
            ))
            trace.append("agenda_agent: completed")
        except CostCapExceededError as exc:
            trace.append(f"cost_cap_guard: aborted - {exc}")
            return PipelineResult(status="aborted_cost_cap", message=str(exc), trace=trace)

        output_check = self._output_guardrail.check(request, venue, av, run_of_show)
        trace.append(f"output_guardrail: passed={output_check.passed} violations={output_check.violations}")
        if not output_check.passed:
            return PipelineResult(
                status="flagged_output",
                message="Plan flagged for human review due to factual-consistency violations.",
                plan=run_of_show,
                violations=output_check.violations,
                trace=trace,
            )

        return PipelineResult(
            status="completed",
            message="Plan assembled and passed all guardrail checks.",
            plan=run_of_show,
            trace=trace,
        )


## 6. Four end-to-end scenarios

| # | Scenario | Raw request | Agent data | Cost cap | Expected `status` |
|---|---|---|---|---|---|
| A | Clean request | On-topic | Consistent, realistic plan | Generous | `completed` |
| B | Misuse | Off-topic ("banana bread recipe") | (irrelevant - never reached) | Generous | `blocked_input` |
| C | Runaway cost | On-topic | Same as A | Tiny (set below the cost of a single call) | `aborted_cost_cap` |
| D | Hallucination | On-topic | Fabricated Miami venue + impossible AV timeline for an Austin request | Generous | `flagged_output` |

All four scenarios share the same `sample_request` (a 120-person Austin, TX conference) - only
the raw text, the scripted agent data, and the cost cap differ, isolating exactly one guard per
scenario (except D, which only needs the output guardrail).


In [10]:
sample_request = EventRequest(
    event_name="Innovate Summit", event_type=EventType.CONFERENCE, headcount=120,
    budget_usd=20000, location="Austin, TX", event_date="2026-09-01",
    start_time="09:00", end_time="15:00", meal_period="lunch",
    dietary_restrictions=["vegetarian"],
)

GOOD_CANNED: Dict[str, Dict[str, Any]] = {
    "venue_recommendation_agent": {
        "recommended_venues": [
            {"name": "Riverside Conference Center", "area": "Downtown Austin", "capacity": 200,
             "estimated_cost_usd": 9000.0, "amenities": ["wifi"], "rationale": "Fits headcount and budget."}
        ],
        "assumptions": [],
    },
    "catering_logistics_agent": {
        "service_style": "buffet",
        "menu_items": [
            {"course": "main", "description": "Herb-roasted chicken", "dietary_tags": []},
            {"course": "main", "description": "Grilled vegetable stack", "dietary_tags": ["vegetarian"]},
        ],
        "staffing_notes": "2 chefs, 4 servers for 120 guests.",
        "estimated_cost_usd": 6000.0, "cost_per_head_usd": 50.0, "assumptions": [],
    },
    "av_vendor_coordination_agent": {
        "av_setup": [{"category": "audio", "description": "Wireless microphones", "quantity": 2}],
        "recommended_vendors": [{"vendor_role": "AV production company", "notes": "Local vendor."}],
        "estimated_cost_usd": 3500.0, "setup_lead_time_hours": 2.0, "assumptions": [],
    },
    "agenda_assembly_agent": {
        "event_name": "Innovate Summit", "event_date": "2026-09-01",
        "selected_venue": "Riverside Conference Center",
        "run_of_show": [
            {"start_time": "07:00", "end_time": "09:00", "activity": "AV setup and sound check", "owner": "AV crew", "location": "Ballroom", "notes": None},
            {"start_time": "09:00", "end_time": "12:00", "activity": "Keynote and breakout sessions", "owner": "Host", "location": "Ballroom", "notes": None},
            {"start_time": "12:00", "end_time": "13:00", "activity": "Buffet lunch service", "owner": "Catering team", "location": "Ballroom", "notes": None},
            {"start_time": "13:00", "end_time": "15:00", "activity": "Afternoon workshops", "owner": "Host", "location": "Ballroom", "notes": None},
        ],
        "overall_notes": ["Vegetarian options confirmed with catering."],
    },
}

# Same catering/AV, but a fabricated Miami venue and an impossible AV-setup timeline.
BAD_CANNED: Dict[str, Dict[str, Any]] = {
    **GOOD_CANNED,
    "venue_recommendation_agent": {
        "recommended_venues": [
            {"name": "South Beach Miami Pavilion", "area": "South Beach, Miami", "capacity": 150,
             "estimated_cost_usd": 9000.0, "amenities": ["ocean view"], "rationale": "Waterfront option."}
        ],
        "assumptions": [],
    },
    "agenda_assembly_agent": {
        "event_name": "Innovate Summit", "event_date": "2026-09-01",
        "selected_venue": "South Beach Miami Pavilion",
        "run_of_show": [
            {"start_time": "09:30", "end_time": "10:00", "activity": "AV setup and sound check", "owner": "AV crew", "location": "Ballroom", "notes": None},
            {"start_time": "09:00", "end_time": "12:00", "activity": "Keynote and breakout sessions", "owner": "Host", "location": "Ballroom", "notes": None},
            {"start_time": "12:00", "end_time": "13:00", "activity": "Buffet lunch service", "owner": "Catering team", "location": "Ballroom", "notes": None},
            {"start_time": "13:00", "end_time": "15:00", "activity": "Afternoon workshops", "owner": "Host", "location": "Ballroom", "notes": None},
        ],
        "overall_notes": ["Vegetarian options confirmed with catering."],
    },
}


def build_pipeline(
    agent_canned: Dict[str, Dict[str, Any]], input_verdict: Dict[str, str], max_cost_usd: float
) -> Tuple[GuardedEventPipeline, KeyedFakeLLMClient, CostCapGuard]:
    """Assemble a fresh `GuardedEventPipeline` for one scenario, so each run gets its own
    cost-cap accumulator and its own scripted input-guardrail verdict."""
    input_guardrail = InputGuardrail(ScriptedLLMClient(responses=[input_verdict]))
    agent_backend = KeyedFakeLLMClient(agent_canned)
    cost_guard = CostCapGuard(agent_backend, max_cost_usd=max_cost_usd)
    pipeline = GuardedEventPipeline(
        venue_agent=VenueAgent(cost_guard),
        catering_agent=CateringLogisticsAgent(cost_guard),
        av_agent=AVVendorCoordinationAgent(cost_guard),
        agenda_agent=AgendaAssemblyAgent(cost_guard),
        input_guardrail=input_guardrail,
        output_guardrail=OutputGuardrail(),
    )
    return pipeline, agent_backend, cost_guard


In [11]:
ON_TOPIC_VERDICT = {
    "category": "on_topic",
    "reason": "Requests sourcing a venue, catering, and AV for a 120-person conference - core event-planning scope.",
}
OFF_TOPIC_VERDICT = {
    "category": "off_topic",
    "reason": "Asks for a banana bread recipe, which has nothing to do with planning a live event.",
}

SCENARIOS = [
    {
        "id": "A",
        "label": "Clean request",
        "raw_text": "We need help organizing a 120-person conference in Austin, TX, including venue, catering, and AV.",
        "input_verdict": ON_TOPIC_VERDICT,
        "agent_canned": GOOD_CANNED,
        "max_cost_usd": 1.00,
    },
    {
        "id": "B",
        "label": "Blocked (off-topic) request",
        "raw_text": "Can you give me a good recipe for banana bread?",
        "input_verdict": OFF_TOPIC_VERDICT,
        "agent_canned": GOOD_CANNED,
        "max_cost_usd": 1.00,
    },
    {
        "id": "C",
        "label": "Cost-cap abort",
        "raw_text": "We need help organizing a 120-person conference in Austin, TX, including venue, catering, and AV.",
        "input_verdict": ON_TOPIC_VERDICT,
        "agent_canned": GOOD_CANNED,
        "max_cost_usd": 0.000001,
    },
    {
        "id": "D",
        "label": "Hallucinated output flagged",
        "raw_text": "We need help organizing a 120-person conference in Austin, TX, including venue, catering, and AV.",
        "input_verdict": ON_TOPIC_VERDICT,
        "agent_canned": BAD_CANNED,
        "max_cost_usd": 1.00,
    },
]

scenario_results = []
for scenario in SCENARIOS:
    pipeline, agent_backend, cost_guard = build_pipeline(
        scenario["agent_canned"], scenario["input_verdict"], scenario["max_cost_usd"]
    )
    result = pipeline.run(scenario["raw_text"], sample_request)
    scenario_results.append({"scenario": scenario, "result": result, "agent_backend": agent_backend, "cost_guard": cost_guard})

    print(f"=== Scenario {scenario['id']}: {scenario['label']} ===")
    print(f"status={result.status}")
    print(f"message={result.message}")
    print(f"agent calls made={len(agent_backend.calls)}  estimated cost so far=${cost_guard.total_cost_usd:.6f}")
    for line in result.trace:
        print(f"  trace: {line}")
    print()

assert scenario_results[0]["result"].status == "completed"
assert scenario_results[1]["result"].status == "blocked_input"
assert len(scenario_results[1]["agent_backend"].calls) == 0, "no agent should be called once input is blocked"
assert scenario_results[2]["result"].status == "aborted_cost_cap"
assert len(scenario_results[2]["agent_backend"].calls) == 0, "cost cap should abort before the inner client is ever called"
assert scenario_results[3]["result"].status == "flagged_output"
assert len(scenario_results[3]["result"].violations) >= 2, "hallucinated plan should trip both rule groups"
print("All four end-to-end guardrail scenarios behaved as expected.")


=== Scenario A: Clean request ===
status=completed
message=Plan assembled and passed all guardrail checks.
agent calls made=4  estimated cost so far=$0.000336
  trace: input_guardrail: category=on_topic reason='Requests sourcing a venue, catering, and AV for a 120-person conference - core event-planning scope.'
  trace: venue_agent: completed
  trace: catering_agent: completed
  trace: av_agent: completed
  trace: agenda_agent: completed
  trace: output_guardrail: passed=True violations=[]

=== Scenario B: Blocked (off-topic) request ===
status=blocked_input
message=Request rejected before any agent ran: Asks for a banana bread recipe, which has nothing to do with planning a live event.
agent calls made=0  estimated cost so far=$0.000000
  trace: input_guardrail: category=off_topic reason='Asks for a banana bread recipe, which has nothing to do with planning a live event.'

=== Scenario C: Cost-cap abort ===
status=aborted_cost_cap
message=We're sorry - completing this request would co

In [12]:
report_rows = []
for entry in scenario_results:
    scenario, result, agent_backend, cost_guard = (
        entry["scenario"], entry["result"], entry["agent_backend"], entry["cost_guard"]
    )
    report_rows.append({
        "Scenario": scenario["id"],
        "Label": scenario["label"],
        "Raw request": scenario["raw_text"][:60] + ("..." if len(scenario["raw_text"]) > 60 else ""),
        "Status": result.status,
        "Agent calls made": len(agent_backend.calls),
        "Estimated cost (USD)": round(cost_guard.total_cost_usd, 6),
        "Violations": "; ".join(result.violations) if result.violations else "-",
        "Message": result.message[:90] + ("..." if len(result.message) > 90 else ""),
    })

report_df = pd.DataFrame(report_rows)
pd.set_option("display.max_colwidth", 90)
report_df


,Scenario,Label,Raw request,Status,Agent calls made,Estimated cost (USD),Violations,Message
0,A,Clean request,"We need help organizing a 120-person conference in Austin, T...",completed,4,0.000336,-,Plan assembled and passed all guardrail checks.
1,B,Blocked (off-topic) request,Can you give me a good recipe for banana bread?,blocked_input,0,0.000000,-,"Request rejected before any agent ran: Asks for a banana bread recipe, which has nothi..."
2,C,Cost-cap abort,"We need help organizing a 120-person conference in Austin, T...",aborted_cost_cap,0,0.000000,-,"We're sorry - completing this request would cost an estimated $0.00038, which exceeds ..."
3,D,Hallucinated output flagged,"We need help organizing a 120-person conference in Austin, T...",flagged_output,4,0.000337,Setup entry 'AV setup and sound check' is scheduled to start (09:30) AFTER the event's...,Plan flagged for human review due to factual-consistency violations.


## Summary

| Deliverable | Where |
|---|---|
| Input guardrail that classifies and blocks off-topic or harmful requests | `InputGuardrail` / `GuardrailVerdict` (Section 2), exercised in Scenario B (`blocked_input`, 0 agent calls) |
| Output validation on at least two factual-consistency rules | `OutputGuardrail` (Section 3): timeline feasibility (overlaps, impossible setup lead time) + venue/location consistency, exercised in Scenario D (`flagged_output`, 2+ violations) |
| Token/cost cap with graceful abort and user-facing message | `CostCapGuard` (Section 4), exercised in Scenario C (`aborted_cost_cap`, 0 agent calls, friendly message before any spend) |

**Result across the four end-to-end scenarios:** Scenario A completes cleanly; Scenario B is
blocked before any agent is invoked; Scenario C is aborted before the first agent call once
projected spend exceeds the configured cap; Scenario D's agents produce a plausible-looking but
hallucinated plan (wrong-city venue + an AV setup that starts after the event begins), which the
output guardrail catches and flags for human review instead of returning silently.
